# Assembly
**Megahit**
https://www.metagenomics.wiki/tools/assembly/megahit
- de novo assembly (w/o reference genome)
- aligns/assembles short reads together to reconstruct one 'metagenome'
- assembled contigs are stored in fasta file

In [22]:
# Using trimmed, qc seqs from /trimmed
# separate into groups based on metadata 
    # spp x health status x sampledata - created in reads_counts. groups found in reads_meta
# 1)remove host from sample reads
# 2)remove symbiont and human reads
# 3)concatenate all f and r seqs into single file (1 for f, 1 for r)
# 4)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, 
    #and ensures there are no gaps - larger portions of genomes if not all are now together in one sequence)

In [23]:
# based on reads_meta groups, make folders and separate samples out

## Metadata and File Setup

In [36]:
import pandas as pd
import numpy as np
import os 
from pathlib import Path

In [25]:
os.chdir("/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw")

In [31]:
reads_meta = pd.read_csv("reads_meta.csv")
reads_meta.head()

,sampleid,raw,trimmed,pct_yield,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,colony_id,group
0,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,4,PSTR,10.0,Diseased_Margin,T1_4_PSTR,52022_PSTR_Diseased_Margin
1,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,12,PSTR,10.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
2,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,4,PSTR,11.0,Diseased_Tissue,T1_4_PSTR,52022_PSTR_Diseased_Tissue
3,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,12,PSTR,11.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
4,052022_BEL_CBC_T1_12_MCAV,166267321,165624856,99.61,52022.0,5/21/22,CBC30N,1.0,8,MCAV,12.0,Diseased_Margin,T1_8_MCAV,52022_MCAV_Diseased_Margin


In [35]:
spp_list=reads_meta['Species'].unique()
print(spp_list)

['PSTR' 'MCAV' 'PAST' 'ORBI' 'MMEA' 'NEG']


In [33]:
reads_meta['group'].unique()

array(['52022_PSTR_Diseased_Margin', '52022_PSTR_Healthy',
       '52022_PSTR_Diseased_Tissue', '52022_MCAV_Diseased_Margin',
       '52022_MCAV_Diseased_Tissue', '52022_PAST_Healthy',
       '52022_OANN_Healthy', '52022_PAST_Diseased_Tissue',
       '52022_MCAV_Healthy', '52022_OFAV_Healthy',
       '52022_PAST_Diseased_Margin', '62019_MMEA_Healthy',
       '62019_PAST_Healthy', '62019_MCAV_Healthy', '102019_PSTR_Healthy',
       '122022_OANN_Diseased_Margin', '122022_PSTR_Healthy',
       '122022_OANN_Healthy', '122022_PSTR_Diseased_Tissue',
       '122022_PSTR_Diseased_Margin', '122022_PAST_Diseased_Margin',
       '122022_OANN_Diseased_Tissue', '122022_PAST_Diseased_Tissue',
       '122022_MCAV_Diseased_Tissue', '122022_PAST_Healthy',
       '122022_MCAV_Healthy', '122022_OFAV_Diseased_Margin',
       '122022_OFAV_Diseased_Tissue', '122022_OFAV_Healthy',
       '122022_MCAV_Diseased_Margin', 'Negative'], dtype=object)

In [ ]:
# make sample list for each spp and group? 

In [ ]:
# common variables to use in scripts
BASE_DIR = "/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw"
STOREREADS = "reads_filtered.txt" # read_count,step,sampleid

In [ ]:
# separating into diff steps 

## SBATCH SCRIPTS

### Coral Host Removal - by Spp

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 168:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assembly-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# set paths for existing bowtie genome indices
MCAV_index=Mcav_DB
MCAV_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mcav_genome/"
MMEA_index=Mmea_DB
MMEA_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mmea_genome/"
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"
# use ssid for past (no PAST host genome - closest relative for genomes we have)
PAST_index=Ssid_DB
PAST_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/"
# use cnat for pstr (no PSTR host genome- closest relative for genomes we have)
PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"
 
# get unique species from list of samples, spp, and groups
spp_list=$(cut -f 2 filtered_sample_groups.txt | tail -n +2 | sort -u)
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# loop through spp list...
for spp in $spp_list; do
    # make spp folder if it doesn't already exist 
    mkdir -p "$spp"

    # identify samples for each spp to host remove together 
    samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

    # copy samples to spp folders
    for id in $samples; do
        if [[ -f "${READSPATH}/${id}_R1_001_val_1.fq" ]] && [[ -f "${READSPATH}/${id}_R2_001_val_2.fq" ]]; then
            # cp "$READSPATH/${id}_R1_001_val_1.fq" "$spp/"
            # cp "$READSPATH/${id}_R2_001_val_2.fq" "$spp/"
            echo "all ${id} files present in $spp"
        else
            echo "Missing files for $spp: $id in $READSPATH"
        fi
    done

    # create file with samplelist and groups for each spp 
    (awk -F'\t' -v s="$spp" '$2 == s' filtered_sample_groups.txt) | cut -f1,3- > $spp/spp_samples 

# 1)remove host from sample reads
# Host seq removal - Thij's script https://github.com/ThijsSt/SCTLD-metagenomes/blob/main/Quality_control_metagenomes.ipynb
    # by specie
    FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
    WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
    mkdir -p $FINALREADS
    mkdir -p $WORKINGPATH

    # assigning path and index variable for each spp       
    spp_index="${spp}_index"
    spp_path="${spp}_path"        
    input_index="${!spp_index}"
    input_path="${!spp_path}"
    
    #skip bowtie index build - already done
    #loop through samples in each spp group
    for id in $samples; do
        #re-align reads back to the index (host genome)
        bowtie2 -p 8 -x $input_path/$input_index -1 $READSPATH/"${id}_R1_001_val_1.fq" -2 $READSPATH/"${id}_R2_001_val_2.fq" -S $WORKINGPATH/"${id}"_mapped_and_unmapped.sam
        
        #convert sam file from bowtie to a bam file for processing
        samtools view -bS $WORKINGPATH/"${id}"_mapped_and_unmapped.sam > $WORKINGPATH/"${id}"_mapped_and_unmapped.bam
        
        #extract only the reads of which both do not match against the host genome
        samtools view -b -f 12 -F 256 $WORKINGPATH/"${id}"_mapped_and_unmapped.bam > $WORKINGPATH/"${id}"_bothReadsUnmapped.bam
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 5G -@ 2 $WORKINGPATH/"${id}"_bothReadsUnmapped.bam -o $WORKINGPATH/"${id}"_bothReadsUnmapped_sorted.bam
        samtools fastq -@ 8 $WORKINGPATH/"${id}"_bothReadsUnmapped_sorted.bam \
            -1 $FINALREADS/"${id}"_host_removed_R1.fastq \
            -2 $FINALREADS/"${id}"_host_removed_R2.fastq \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    done     
done
conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53514255
# bash script file name: host_removal

In [ ]:
# run multiple scripts for multiple spp at the same time to make faster 
# start with orbi for second script

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyORBI-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# set paths for spp & existing bowtie genome indices
spp="ORBI" 
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707659,53723284
# bash script file name: host_removal_orbi

In [ ]:
# repeat PAST

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="PAST" 
# use ssid for past (no PAST host genome - closest relative for genomes we have)
PAST_index=Ssid_DB
PAST_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707344, 53723373
# bash script file name: host_removal_past

In [ ]:
# repeat pstr

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="PSTR" 
# use cnat for pstr (no PSTR host genome- closest relative for genomes we have)
PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707348, 53723406
# bash script file name: host_removal_pstr

In [ ]:
# rerun mmea bc original script failed in the middle due to time clock

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="MMEA" 
MMEA_index=Mmea_DB
MMEA_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mmea_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53723424
# bash script file name: host_removal_mmea

In [ ]:
# will run read count scripts separately 

### Symbiont Removal and Assembly - by Spp

In [ ]:
# remove symbionts, finish assembly step 
# run for each spp, and loop through groups within spp for assembly step

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-mcav_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load Bowtie2/2.4.5-GCC-11.3.0
module load conda/latest
#conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="MCAV"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

#mkdir -p "$OUTPUTDIR"
# --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
#while IFS=$'\t' read -r SAMPLEID GROUP; do
#    $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
#    $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq;
#     if [ $? -eq 0 ]; then
#            echo "fastq_screen completed successfully for sample: $SAMPLEID"
#        else
#            echo "fastq_screen encountered an error for sample: $SAMPLEID"
#            exit 1
#        fi
#done < $SAMPLEFILE
#conda deactivate
#echo "Symbiont, human removal: All samples processed successfully."

#From Nikea: Using repair.sh script from:https://jgi.doe.gov/data-and-tools/software-tools/bbtools/bb-tools-user-guide/repair-guide/
# re-pair scripts after fastqscreen - sometimes paired reads get unpaired during the symbiont removal phase
# make sure to install first in conda assembly env
conda activate assembly
# conda install -c bioconda bbmap

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$READSPATH"
gzip *.fastq

while IFS=$'\t' read -r SAMPLEID GROUP; do
    repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
    out1=$OUTDIR/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=$OUTDIR/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
    outs=$OUTDIR/"${SAMPLEID}"singletons.fq repair;
     if [ $? -eq 0 ]; then
            echo "repair completed successfully for sample: $SAMPLEID"
        else
            echo "repair encountered an error for sample: $SAMPLEID"
            exit 1
        fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
# redo and use repaired
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    rm -rf "$OUTDIR"    # delete and redo 
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done
conda deactivate 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"
    rm -rf "$MEG_OUT"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
# use

# JOB-ID: 53923368, 54683562, 54750736
# bash script file name: mcav_assembly2

In [ ]:
# skip mmea for now - input files are in the process of being zipped 

#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-mmea_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="MMEA"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

mkdir -p "$OUTPUTDIR"
# --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
while IFS=$'\t' read -r SAMPLEID GROUP; do
    $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
    $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
     if [ $? -eq 0 ]; then
            echo "fastq_screen completed successfully for sample: $SAMPLEID"
        else
            echo "fastq_screen encountered an error for sample: $SAMPLEID"
            exit 1
        fi
done < $SAMPLEFILE
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter.fastq" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter.fastq" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter.fastq|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter.fastq|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done
conda deactivate 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for group in $spp_groups; do
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    megahit --presets meta-large \
    -1 "$OUTDIR"/"$group"_reads_R1_ALL.fastq.gz \
    -2 "$OUTDIR"/"$group"_reads_R2_ALL.fastq.gz \
    --keep-tmp-files \
    -o megahit_host_removed --out-prefix $group \
    --continue
done
# megahit has to make the directory; will fail if it already exists

# JOB-ID: 54684038
# bash script file name: mmea_assembly2

In [ ]:
## ORBI

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-orbi_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
# conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="ORBI"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

# mkdir -p "$OUTPUTDIR"
# # zip any unzipped host_removed files - idk what happened in previous step 
# cd "$READSPATH"
# gzip *.fastq
# # --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
# while IFS=$'\t' read -r SAMPLEID GROUP; do
#     $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
#     $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
#      if [ $? -eq 0 ]; then
#             echo "fastq_screen completed successfully for sample: $SAMPLEID"
#         else
#             echo "fastq_screen encountered an error for sample: $SAMPLEID"
#             exit 1
#         fi
# done < "$SAMPLEFILE"
# conda deactivate
# echo "Symbiont, human removal: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done
conda deactivate 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for group in $spp_groups; do
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    megahit --presets meta-large \
    -1 "$OUTDIR"/"$group"_reads_R1_ALL.fastq.gz \
    -2 "$OUTDIR"/"$group"_reads_R2_ALL.fastq.gz \
    --keep-tmp-files \
    -o megahit_host_removed --out-prefix $group \
    --continue
done
# megahit has to make the directory; will fail if it already exists

# JOB-ID: 53971713, 54693815
# bash script file name: orbi_assembly2

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-PAST_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PAST"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

mkdir -p "$OUTPUTDIR"
# zip any unzipped host_removed files - idk what happened in previous step 
cd "$READSPATH"
# gzip *.fastq

# --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
while IFS=$'\t' read -r SAMPLEID GROUP; do
    # check if file exists first since first script run failed halfway through
    CHECK_FILE="${READSPATH}/${SAMPLEID}_host_removed_R1.tagged_filter.fastq"
    if [ -f "$CHECK_FILE" ]; then
        echo "Sample $SAMPLEID already processed. Skipping..."
        continue
    fi
    
    echo "Processing sample: $SAMPLEID"
    $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
    $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;

    if [ $? -eq 0 ]; then
            echo "fastq_screen completed successfully for sample: $SAMPLEID"
    else
            echo "fastq_screen encountered an error for sample: $SAMPLEID"
            exit 1
    fi
done < "$SAMPLEFILE"
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done
conda deactivate 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for group in $spp_groups; do
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    megahit --presets meta-large \
    -1 "$OUTDIR"/"$group"_reads_R1_ALL.fastq.gz \
    -2 "$OUTDIR"/"$group"_reads_R2_ALL.fastq.gz \
    --keep-tmp-files \
    -o megahit_host_removed --out-prefix $group \
    --continue
done
# megahit has to make the directory; will fail if it already exists

# JOB-ID: 53972006, 54694069
# bash script file name: PAST_assembly2

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-PSTR_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PSTR"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

mkdir -p "$OUTPUTDIR"
# zip any unzipped host_removed files - idk what happened in previous step 
cd "$READSPATH"
# gzip *.fastq
# --nohits = output reads do not map to any genomes - removes human and symbiont seq matches

while IFS=$'\t' read -r SAMPLEID GROUP; do
    # check if file exists first since first script run failed halfway through
    CHECK_FILE="${READSPATH}/${SAMPLEID}_host_removed_R1.tagged_filter.fastq"
    if [ -f "$CHECK_FILE" ]; then
        echo "Sample $SAMPLEID already processed. Skipping..."
        continue
    fi
    
    echo "Processing sample: $SAMPLEID"
    $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
    $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;

    if [ $? -eq 0 ]; then
            echo "fastq_screen completed successfully for sample: $SAMPLEID"
    else
            echo "fastq_screen encountered an error for sample: $SAMPLEID"
            exit 1
    fi
done < "$SAMPLEFILE"
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.fastq.tagged_filter.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.fastq.tagged_filter.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.fastq.tagged_filter.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.fastq.tagged_filter.fastq.gz|" | xargs cat | gzip -c > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done
conda deactivate 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for group in $spp_groups; do
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    megahit --presets meta-large \
    -1 "$OUTDIR"/"$group"_reads_R1_ALL.fastq.gz \
    -2 "$OUTDIR"/"$group"_reads_R2_ALL.fastq.gz \
    --keep-tmp-files \
    -o megahit_host_removed --out-prefix $group \
    --continue
done
# megahit has to make the directory; will fail if it already exists

# JOB-ID: 53972008, 54694110
# bash script file name: PSTR_assembly2

In [ ]:
# below is the fastqscreen.conf file 

In [ ]:
############################
## Bowtie, Bowtie 2 or BWA #
############################
## If the Bowtie, Bowtie 2 or BWA binary is not in your PATH, you can set 
## this value to tell the program where to find your chosen aligner.  Uncomment 
## the relevant line below and set the appropriate location.  Please note, 
## this path should INCLUDE the executable filename.

#BOWTIE	/usr/local/bin/bowtie/bowtie
#BOWTIE2 /usr/local/bowtie2/bowtie2
#BWA /usr/local/bwa/bwa

############################################
## Bismark (for bisulfite sequencing only) #
############################################
## If the Bismark binary is not in your PATH then you can set this value to 
## tell the program where to find it.  Uncomment the line below and set the 
## appropriate location. Please note, this path should INCLUDE the executable 
## filename.

#BISMARK	/usr/local/bin/bismark/bismark

############
## Threads #
############
## Genome aligners can be made to run across multiple CPU cores to speed up 
## searches.  Set this value to the number of cores you want for mapping reads.

THREADS		12

##############
## DATABASES #
##############
## This section enables you to configure multiple genomes databases (aligner index 
## files) to search against in your screen.  For each genome you need to provide a 
## database name (which can't contain spaces) and the location of the aligner index 
## files.
##
## The path to the index files SHOULD INCLUDE THE BASENAME of the index, e.g:
## /data/public/Genomes/Human_Bowtie/GRCh37/Homo_sapiens.GRCh37
## Thus, the index files (Homo_sapiens.GRCh37.1.bt2, Homo_sapiens.GRCh37.2.bt2, etc.) 
## are found in a folder named 'GRCh37'.
##
## If, for example, the Bowtie, Bowtie2 and BWA indices of a given genome reside in 
## the SAME FOLDER, a SINLGE path may be provided to ALL the of indices.  The index 
## used will be the one compatible with the chosen aligner (as specified using the 
## --aligner flag).  
##
## The entries shown below are only suggested examples, you can add as many DATABASE 
## sections as required, and you can comment out or remove as many of the existing 
## entries as desired.  We suggest including genomes and sequences that may be sources 
## of contamination either because they where run on your sequencer previously, or may 
## have contaminated your sample during the library preparation step.
##
## Human - sequences available from
## ftp://ftp.ensembl.org/pub/current/fasta/homo_sapiens/dna/
## (Kraken2 RefSeq db)
DATABASE	Human	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ref_databases/standard/library/human/index
##
## added more databases and updated a few listed here with their updated assemblies 12.11.2024
## Symbionts
DATABASE	Symbiont1	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/Durusdinium_trenchii_indexed
## Symbionts
DATABASE	Symbiont2	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_000507305.1_index
## Symbionts
DATABASE	Symbiont3	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_001939145.1_index
## Symbionts
DATABASE	Symbiont4	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_003297005.1_index
## Symbionts
DATABASE	Symbiont5	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_009767595.1_index
## Symbionts
DATABASE	Symbiont6	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_018327485.1_index
## Symbionts
DATABASE	Symbiont7	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_905221635.1_index
## Symbionts
DATABASE	Symbiont8	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_947184155.2_index
## Symbionts
DATABASE	Symbiont9	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_003297045.1_index
## Symbionts
DATABASE	Symbiont10	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_905231905.1_index
## Symbionts
DATABASE	Symbiont11	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_905231915.1_index
##
## Ecoli- sequence available from EMBL accession U00096.2
#DATABASE	Ecoli	/data/public/Genomes/Ecoli/Ecoli
##
## PhiX - sequence available from Refseq accession NC_001422.1
#DATABASE	PhiX	/data/public/Genomes/PhiX/phi_plus_SNPs
##
## Adapters - sequence derived from the FastQC contaminats file found at: www.bioinformatics.babraham.ac.uk/projects/fastqc
#DATABASE	Adapters	/data/public/Genomes/Contaminants/Contaminants
##
## Vector - Sequence taken from the UniVec database
## http://www.ncbi.nlm.nih.gov/VecScreen/UniVec.html
#DATABASE	Vectors		/data/public/Genomes/Vectors/Vectors

In [ ]:
# make script to zip all intermediate files
# SPP= 
# OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
# TAGGEDDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed/symb_seqs"

# # go ahead and zip tagged.fastq files 
# mkdir -p "$TAGGEDDIR"
# cd "$OUTPUTDIR"

# rsync -av "$OUTPUTDIR" "$TAGGEDDIR"
# ZSTD_NBTHREADS=0 tar --zstd -cf raw.tar.zst raw

# if [ $? -eq 0 ]; then
#    echo "Zipped "raw" successfully!"
# else
#    echo "Command failed! (zipping "raw")"
# fi
# # test integrity of zipped dir
# zstd -t raw.tar.zst
# if [ $? -eq 0 ]; then
#    echo "Integrity check passed! Deleting uncompressed copy at destination..."
#    rm -rf /scratch/workspace/brooke_sienkiewicz_student_uml_edu-raw_seqs/raw
# else
#    echo "CRITICAL ERROR: Integrity check failed. Keeping uncompressed 'raw' for safety."
#    exit 1
# fi